# Продуктовый кейс `roza4u.ru`: юнит-экономика и дизайн A/B-теста конструктора букетов

**Автор:** Юлия Богодяж  
**Цель ноутбука:** провести количественную оценку внедрения функции «Собери свой букет» (модель юнит-экономики в 3 сценариях), рассчитать параметры A/B-теста (`sample size`, `MDE`, длительность) и выполнить предэкспериментальную валидацию статистического дизайна (A/A-симуляция и оценка мощности).

> **Важно:** все симуляции в разделе 3 носят характер *предэкспериментальной проверки дизайна* (до запуска реального A/B-теста в продакшене).

In [1]:
import math
from statistics import NormalDist
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

NORM = NormalDist(0.0, 1.0)

# Базовые параметры бизнеса (допущения на 1 месяц)
MONTHLY_VISITORS = 100_000
WEEKLY_VISITORS = MONTHLY_VISITORS * 7 / 30  # ~23 333 пользователя/нед
BASELINE_CR = 0.03
BASELINE_AOV = 4_500.0
GROSS_MARGIN_RATE = 0.45
EXTRA_ASSEMBLY_COST = 120.0   # доп. затраты на индивидуальную сборку/упаковку
MVP_DEV_COST = 450_000.0      # оценка разовой разработки MVP

baseline_orders = MONTHLY_VISITORS * BASELINE_CR
baseline_revenue = baseline_orders * BASELINE_AOV
baseline_gp = baseline_revenue * GROSS_MARGIN_RATE
print(f'Базовая выручка: {baseline_revenue:,.0f} ₽/мес | Валовая прибыль: {baseline_gp:,.0f} ₽/мес')


Базовая выручка: 13,500,000 ₽/мес | Валовая прибыль: 6,075,000 ₽/мес


## 1. Модель юнит-экономики и анализ чувствительности (3 сценария)

> **Важно о данных:** поскольку доступа к внутренней базе `roza4u.ru` нет, все входные параметры ниже (трафик, базовая конверсия, маржа 45%, доп. затраты на сборку 120 ₽, стоимость MVP 450 тыс. ₽ и диапазон `adoption` 15–25%) — **гипотетические допущения для сценарного моделирования**. В реальной компании эти числа берутся из SQL-логов, отчётов финансов и оценок разработки.

Вместо допущения, что конструктор букетов мгновенно изменит метрики у 100% посетителей сайта, эффект декомпозируется через **долю пользователей фичи (`adoption`)**:

- Посетители, не открывшие конструктор ($1 - a$), сохраняют базовые `CR = 3.0%` и `AOV = 4 500 ₽`.
- Посетители, открывшие конструктор ($a$), формируют отдельный поток заказов с целевыми `CR_builder` и `AOV_builder`, причём на каждый кастомный букет закладываются дополнительные операционные затраты `120 ₽` (время флориста и индивидуальная упаковка).

In [2]:
configs = [
    {'scenario': 'Пессимистичный', 'adoption': 0.15, 'cr_builder': 0.038, 'aov_builder': 4_900.0},
    {'scenario': 'Базовый',        'adoption': 0.20, 'cr_builder': 0.045, 'aov_builder': 5_200.0},
    {'scenario': 'Оптимистичный',  'adoption': 0.25, 'cr_builder': 0.046, 'aov_builder': 5_500.0},
]

rows = []
for cfg in configs:
    a = cfg['adoption']
    orders_reg = MONTHLY_VISITORS * (1 - a) * BASELINE_CR
    orders_bld = MONTHLY_VISITORS * a * cfg['cr_builder']
    tot_orders = orders_reg + orders_bld
    tot_rev = orders_reg * BASELINE_AOV + orders_bld * cfg['aov_builder']
    tot_gp = tot_rev * GROSS_MARGIN_RATE - orders_bld * EXTRA_ASSEMBLY_COST
    delta_rev = tot_rev - baseline_revenue
    delta_gp = tot_gp - baseline_gp
    rows.append({
        'Сценарий': cfg['scenario'],
        'Adoption (%)': round(a * 100, 1),
        'CR в конструкторе (%)': round(cfg['cr_builder'] * 100, 2),
        'Общий CR (%)': round(tot_orders / MONTHLY_VISITORS * 100, 2),
        'Δ CR (п.п.)': round((tot_orders / MONTHLY_VISITORS - BASELINE_CR) * 100, 2),
        'Чек конструктора (₽)': int(cfg['aov_builder']),
        'Общий AOV (₽)': round(tot_rev / tot_orders),
        'ARPU (₽)': round(tot_rev / MONTHLY_VISITORS, 2),
        'Выручка/мес (₽)': round(tot_rev),
        'Δ Выручки/мес (₽)': round(delta_rev),
        'Δ Валовой прибыли/мес (₽)': round(delta_gp),
        'Окупаемость MVP (мес.)': round(MVP_DEV_COST / delta_gp, 2),
        'ROI за 6 мес. (%)': round((delta_gp * 6 - MVP_DEV_COST) / MVP_DEV_COST * 100),
    })

df_scenarios = pd.DataFrame(rows)
df_scenarios


Сценарий,Adoption (%),CR в конструкторе (%),Общий CR (%),Δ CR (п.п.),Чек конструктора (₽),Общий AOV (₽),ARPU (₽),Выручка/мес (₽),Δ Выручки/мес (₽),Δ Валовой прибыли/мес (₽),Окупаемость MVP (мес.),ROI за 6 мес. (%)
Пессимистичный,15.0,3.8,3.12,0.12,4900,4573,142.68,14268000,768000,277200,1.62,270
Базовый,20.0,4.5,3.30,0.30,5200,4691,154.80,15480000,1980000,783000,0.57,944
Оптимистичный,25.0,4.6,3.40,0.40,5500,4838,164.50,16450000,2950000,1189500,0.38,1486


In [3]:
fig_ue = go.Figure()
fig_ue.add_bar(name='Δ Выручки / мес (млн ₽)', x=df_scenarios['Сценарий'], y=df_scenarios['Δ Выручки/мес (₽)'] / 1e6, marker_color='#E397B4')
fig_ue.add_bar(name='Δ Валовой прибыли / мес (млн ₽)', x=df_scenarios['Сценарий'], y=df_scenarios['Δ Валовой прибыли/мес (₽)'] / 1e6, marker_color='#A63A66')
fig_ue.update_layout(
    title='Прирост выручки и валовой прибыли по сценариям',
    yaxis_title='Млн ₽ / мес',
    barmode='group',
    template='plotly_white',
)
fig_ue.show()


## 2. Расчёт размера выборки (Sample Size), MDE и длительности A/B-теста

Для ключевой бинарной метрики **CR в оплаченный заказ** (двусторонний z-тест пропорций, $\alpha = 5\%$, мощность $80\%$):

In [4]:
def sample_size_proportion(p1: float, p2: float, alpha: float = 0.05, power: float = 0.80) -> int:
    z_alpha = NORM.inv_cdf(1.0 - alpha / 2.0)
    z_beta = NORM.inv_cdf(power)
    p_bar = (p1 + p2) / 2.0
    num = (z_alpha * math.sqrt(2 * p_bar * (1 - p_bar)) + z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return math.ceil(num / ((p2 - p1) ** 2))

for name, target_cr in [('Оптимистичный (+0.40 п.п.)', 0.034), ('Базовый (+0.30 п.п.)', 0.033), ('Пессимистичный (+0.12 п.п.)', 0.0312)]:
    n = sample_size_proportion(BASELINE_CR, target_cr)
    weeks = (2 * n) / WEEKLY_VISITORS
    print(f'{name:28s}: n = {n:>7,} чел./группу | всего {2*n:>7,} | длительность ≈ {weeks:.1f} нед.')


Оптимистичный (+0.40 п.п.)  : n =  30,390 чел./группу | всего  60,780 | длительность ≈ 2.6 нед.
Базовый (+0.30 п.п.)        : n =  53,211 чел./группу | всего 106,422 | длительность ≈ 4.6 нед.
Пессимистичный (+0.12 п.п.) : n = 323,369 чел./группу | всего 646,738 | длительность ≈ 27.7 нед.


In [5]:
def mde_for_n(p1: float, n_per_group: float, alpha: float = 0.05, power: float = 0.80) -> float:
    lo, hi = 1e-5, 0.20
    for _ in range(60):
        mid = (lo + hi) / 2.0
        if sample_size_proportion(p1, p1 + mid, alpha, power) > n_per_group:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2.0

mde_rows = []
for w in range(1, 7):
    tot = round(WEEKLY_VISITORS * w)
    n_g = tot // 2
    mde_abs = mde_for_n(BASELINE_CR, n_g)
    mde_rows.append({
        'Недели': w,
        'Всего пользователей': tot,
        'В каждой группе': n_g,
        'MDE (п.п.)': round(mde_abs * 100, 2),
        'MDE (% отн.)': round(mde_abs / BASELINE_CR * 100, 1),
        'Целевой CR (%)': round((BASELINE_CR + mde_abs) * 100, 2),
    })
df_mde = pd.DataFrame(mde_rows)
df_mde


Недели,Всего пользователей,В каждой группе,MDE (п.п.),MDE (% отн.),Целевой CR (%)
1,23333,11666,0.66,21.9,3.66
2,46667,23333,0.46,15.3,3.46
3,70000,35000,0.37,12.4,3.37
4,93333,46666,0.32,10.7,3.32
5,116667,58333,0.29,9.5,3.29
6,140000,70000,0.26,8.7,3.26


In [6]:
fig_mde = go.Figure()
fig_mde.add_scatter(x=df_mde['Недели'], y=df_mde['MDE (п.п.)'], mode='lines+markers+text',
                    text=[f'+{v:.2f}' for v in df_mde['MDE (п.п.)']], textposition='top center',
                    line=dict(color='#A63A66', width=3), name='MDE (п.п.)')
fig_mde.add_hline(y=0.40, line_dash='dash', line_color='#C25982', annotation_text='MDE +0.40 п.п. (~3 нед.)')
fig_mde.add_hline(y=0.30, line_dash='dash', line_color='#2E7D6F', annotation_text='Базовый эффект +0.30 п.п. (~5 нед.)')
fig_mde.update_layout(title='MDE по конверсии в зависимости от длительности теста',
                      xaxis_title='Длительность (недели)', yaxis_title='MDE (п.п.)', template='plotly_white')
fig_mde.show()


## 3. Предэкспериментальная валидация дизайна (A/A-симуляция и оценка мощности)

Перед запуском эксперимента проверяем корректность статистического критерия методом Монте-Карло:
1. **A/A-симуляция ($H_0$ верна):** распределение p-value должно быть равномерным $U(0, 1)$, а доля ложноположительных срабатываний (FPR) — близка к $\alpha = 5\%$.
2. **A/B-симуляция ($H_1$ верна: CR 3.0% → 3.4% при $n = 30\,400$):** эмпирическая мощность должна подтверждать расчётные $\approx 80\%$.

In [7]:
rng = np.random.default_rng(42)
N_GROUP = 30_400
N_SIMS = 2_000

def run_z_batch(p_a: float, p_b: float) -> np.ndarray:
    x_a = rng.binomial(N_GROUP, p_a, size=N_SIMS)
    x_b = rng.binomial(N_GROUP, p_b, size=N_SIMS)
    p_pool = (x_a + x_b) / (2 * N_GROUP)
    se = np.sqrt(p_pool * (1 - p_pool) * (2 / N_GROUP))
    z = np.abs(x_b / N_GROUP - x_a / N_GROUP) / se
    return np.array([math.erfc(val / math.sqrt(2)) for val in z])

aa_pvals = run_z_batch(0.03, 0.03)
ab_pvals = run_z_batch(0.03, 0.034)

print(f'A/A-симуляция — эмпирический FPR (α=5%): {np.mean(aa_pvals < 0.05)*100:.2f}%')
print(f'A/B-симуляция — эмпирическая мощность (цель 80%): {np.mean(ab_pvals < 0.05)*100:.1f}%')


A/A-симуляция — эмпирический FPR (α=5%): 5.05%
A/B-симуляция — эмпирическая мощность (цель 80%): 79.9%


In [8]:
fig_val = make_subplots(rows=1, cols=2, subplot_titles=('A/A-симуляция (p-value, H0)', 'A/B-симуляция (p-value, H1: +0.40 п.п.)'))
fig_val.add_histogram(x=aa_pvals, nbinsx=10, histnorm='percent', marker_color='#E397B4', row=1, col=1, showlegend=False)
fig_val.add_histogram(x=ab_pvals, nbinsx=10, histnorm='percent', marker_color='#A63A66', row=1, col=2, showlegend=False)
fig_val.update_layout(title='Валидация статистического критерия до запуска эксперимента', template='plotly_white')
fig_val.show()


## Выводы

1. **Экономика фичи устойчива даже в пессимистичном сценарии:** при adoption 15% и умеренном росте CR до 3.12% дополнительная валовая прибыль составляет **+277 тыс. ₽/мес**, а MVP за 450 тыс. ₽ окупается за **~1.6 месяца**. В базовом сценарии (adoption 20%, общий CR 3.30%) прирост валовой прибыли достигает **+783 тыс. ₽/мес** (окупаемость ~0.6 мес.).
2. **Длительность A/B-теста:** при трафике 100 000 пользователей/мес тест на ~30 400 пользователей в группе (MDE +0.40 п.п.) требует **3 полных недели** (а не 2). Чтобы уверенно поймать эффект базового сценария (+0.30 п.п.), понадобится **5 недель** (дополнительно оцениваем денежные метрики `ARPU` и `AOV` через t-тест Уэлча / бутстреп с контролем выбросов).
3. **Критерий валидирован:** A/A-симуляция подтвердила контроль ошибки I рода на уровне ~5%, а A/B-симуляция — расчётную мощность ~80%.